# 03 — Customer Lifetime Value Prediction
BG/NBD + Gamma-Gamma model for 12-month CLV.

In [ ]:
import sys; sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from src.clv import load_transactions, build_rfm_summary, fit_bgf, fit_ggf, predict_clv
import warnings; warnings.filterwarnings('ignore')

engine = create_engine('sqlite:///../data/ecommerce.db')
sns.set_theme(style='whitegrid')
print("Ready")

## 1. What is CLV?
Customer Lifetime Value = how much revenue a customer will generate in the future.

We use two probabilistic models:
- **BG/NBD** (Beta-Geometric / Negative Binomial Distribution): models *when* a customer makes their next purchase
- **Gamma-Gamma**: models *how much* they'll spend per transaction

## 2. Build Lifetimes Summary

In [ ]:
df      = load_transactions(engine)
summary = build_rfm_summary(df)
print(f"Repeat customers: {len(summary):,}")
print("\nFirst 5 rows:")
summary.head()

## 3. Fit BG/NBD (Purchase Frequency Model)

In [ ]:
bgf = fit_bgf(summary)

# Visualise: expected purchases in next 26 weeks for a range of R/F values
t_values = [1, 7, 14, 26]
fig, ax = plt.subplots(figsize=(10,5))
for t in t_values:
    x = np.linspace(1, 15, 50)
    y = [bgf.conditional_expected_number_of_purchases_up_to_time(
             t, xi, float(summary['recency'].mean()), float(summary['T'].mean()))
         for xi in x]
    ax.plot(x, y, lw=2, label=f'{t} weeks')
ax.set_xlabel('Historical Frequency'); ax.set_ylabel('Expected Future Purchases')
ax.set_title('BG/NBD: Expected Future Purchases by Horizon')
ax.legend(); plt.tight_layout(); plt.show()

## 4. Fit Gamma-Gamma (Spend Model)

In [ ]:
ggf = fit_ggf(summary)
pred_avg = ggf.conditional_expected_average_profit(summary['frequency'], summary['monetary_value'])
actual    = summary['monetary_value']

fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(actual, pred_avg, alpha=0.3, s=10, color='#2980b9')
lims = [0, min(actual.quantile(0.95), pred_avg.quantile(0.95))*1.1]
ax.plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
ax.set_xlabel('Actual Avg Transaction Value (R$)'); ax.set_ylabel('Predicted (R$)')
ax.set_title('Gamma-Gamma: Predicted vs Actual Avg Spend')
ax.legend(); ax.set_xlim(*lims); ax.set_ylim(*lims)
plt.tight_layout(); plt.show()

## 5. Predict 12-Month CLV

In [ ]:
clv_df = predict_clv(summary.copy(), bgf, ggf, months=12)
print(clv_df[['customer_id','frequency','monetary_value','prob_alive','clv_12m','clv_tier']].head(10).round(2).to_string(index=False))

## 6. CLV by Tier

In [ ]:
tier_summary = clv_df.groupby('clv_tier').agg(
    customers=('customer_id','count'),
    avg_clv=('clv_12m','mean'),
    total_clv=('clv_12m','sum'),
    avg_prob_alive=('prob_alive','mean')
).round(2)
print(tier_summary)

fig, axes = plt.subplots(1,2,figsize=(14,5))
tier_colors = {'Low Value':'#e74c3c','Mid Value':'#f39c12','High Value':'#2ecc71'}
for tier in clv_df['clv_tier'].unique():
    grp = clv_df[clv_df['clv_tier']==tier]
    axes[0].hist(grp['clv_12m'].clip(upper=grp['clv_12m'].quantile(0.95)),
                 bins=40, alpha=0.7, label=str(tier), color=tier_colors.get(str(tier),'#888'))
axes[0].set_title('CLV Distribution by Tier'); axes[0].set_xlabel('12M CLV (R$)'); axes[0].legend()

total_by_tier = tier_summary['total_clv']
axes[1].pie(total_by_tier, labels=total_by_tier.index, autopct='%1.1f%%',
            colors=[tier_colors.get(str(t),'#888') for t in total_by_tier.index])
axes[1].set_title('Total Projected Revenue by CLV Tier')
plt.tight_layout(); plt.show()

## 7. Export

In [ ]:
clv_df.to_csv('../outputs/clv_predictions.csv', index=False)
print(f"Exported {len(clv_df):,} customer CLV predictions")
print(f"\nAvg 12M CLV: R$ {clv_df['clv_12m'].mean():,.2f}")
print(f"Total projected 12M revenue: R$ {clv_df['clv_12m'].sum():,.0f}")